# SeedSec Rwanda: Vegetable Seed Image Classification Training Notebook
This Jupyter notebook contains the training script to train a **MobileNetV2** model for classifying **14 classes of vegetable seeds** using the Kaggle dataset. It handles mounting Google Drive, setting up API credentials, downloading the dataset, splitting it into training, validation, and test subsets, training the classification model with PyTorch/torchvision, validating performance (F1-score, precision, recall, and plotting curves), and exporting the weights to quantized TFLite format.

## Step 1: Set Kaggle Token & Mount Google Drive

In [ ]:
import os
import glob
import random
import shutil
from pathlib import Path

source_data_dir = Path("/content/vegetable_dataset")
split_target_dir = Path("/content/vegetable_splits")

# Define splits and make sure they exist
splits = ["train", "val", "test"]
for s in splits:
    (split_target_dir / s).mkdir(parents=True, exist_ok=True)

# Find all images recursively and identify their class directories
valid_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
image_files = []
for ext in valid_extensions:
    image_files.extend(source_data_dir.glob(f"**/*{ext}"))
    image_files.extend(source_data_dir.glob(f"**/*{ext.upper()}"))

# Group images by their class (the name of the directory they reside in)
# We exclude directory names that are split names (train, val, test, validation)
class_to_images = {}
for img_path in image_files:
    parent_dir = img_path.parent
    class_name = parent_dir.name
    if class_name.lower() in ["train", "val", "test", "validation", "vegetable_splits"]:
        continue
    if class_name not in class_to_images:
        class_to_images[class_name] = []
    class_to_images[class_name].append(img_path)

print(f"Found {len(class_to_images)} class folders:")
for class_name, imgs in class_to_images.items():
    print(f" - {class_name} ({len(imgs)} images)")

# Perform split
random.seed(42)
for class_name, all_imgs in class_to_images.items():
    random.shuffle(all_imgs)
    
    # Create splits directories
    for s in splits:
        (split_target_dir / s / class_name).mkdir(parents=True, exist_ok=True)
        
    n_total = len(all_imgs)
    n_train = int(n_total * 0.70)
    n_val = int(n_total * 0.15)
    
    train_imgs = all_imgs[:n_train]
    val_imgs = all_imgs[n_train:n_train + n_val]
    test_imgs = all_imgs[n_train + n_val:]
    
    for img in train_imgs:
        shutil.copy(str(img), split_target_dir / "train" / class_name / img.name)
    for img in val_imgs:
        shutil.copy(str(img), split_target_dir / "val" / class_name / img.name)
    for img in test_imgs:
        shutil.copy(str(img), split_target_dir / "test" / class_name / img.name)

print("Splits successfully configured:")
for s in splits:
    total_count = len(list((split_target_dir / s).glob("**/*.*")))
    print(f" - {s} subset size: {total_count} images")

## Step 2: Download the Vegetable Seed Dataset from Kaggle

In [ ]:
# Install Kaggle python package
!pip install kaggle -q

# Create target workspace directories on Drive
!mkdir -p "/content/drive/MyDrive/KaggleDatasets/vegetable-seed-dataset"

# Download the dataset archive
!kaggle datasets download -d imr4n4lif/vegetable-seed-dataset \
  -p "/content/drive/MyDrive/KaggleDatasets/vegetable-seed-dataset"

# Unzip files to Google Colab local runtime for fast read access
print("Extracting dataset to local storage...")
!unzip -q "/content/drive/MyDrive/KaggleDatasets/vegetable-seed-dataset/vegetable-seed-dataset.zip" -d "/content/vegetable_dataset"
print("Extraction complete.")

## Step 3: Organize Images & Train/Val/Test Split
Scans the directory structure of the extracted Kaggle dataset, identifies all class folders (14 types of vegetable seeds), and splits the images into train (70%), validation (15%), and test (15%) splits.

In [ ]:
import os
import glob
import random
import shutil
from pathlib import Path

source_data_dir = Path("/content/vegetable_dataset")
split_target_dir = Path("/content/vegetable_splits")

# Define splits
splits = ["train", "val", "test"]
for s in splits:
    split_target_dir.mkdir(parents=True, exist_ok=True)

# Locate class directories
# Adjust folder paths to support nested folder extraction from zip file
subdirs = [x for x in source_data_dir.glob("**/*") if x.is_dir()]
# Find directories that directly contain images (e.g., .jpg, .png)
class_dirs = []
for d in subdirs:
    images = glob.glob(str(d / "*.jpg")) + glob.glob(str(d / "*.png")) + glob.glob(str(d / "*.jpeg"))
    if len(images) > 0 and d.name.lower() not in ["train", "val", "test", "validation"]:
        class_dirs.append(d)

# Remove duplicates by taking unique path leaves
class_dirs = list({d.name: d for d in class_dirs}.values())

print(f"Found {len(class_dirs)} class folders:")
for d in class_dirs:
    print(f" - {d.name} ({len(glob.glob(str(d / '*')))} files)")

# Perform split
random.seed(42)
for c_dir in class_dirs:
    class_name = c_dir.name
    all_imgs = sorted(list(c_dir.glob("*.jpg")) + list(c_dir.glob("*.png")) + list(c_dir.glob("*.jpeg")))
    random.shuffle(all_imgs)
    
    # Create splits directories
    for s in splits:
        (split_target_dir / s / class_name).mkdir(parents=True, exist_ok=True)
        
    n_total = len(all_imgs)
    n_train = int(n_total * 0.70)
    n_val = int(n_total * 0.15)
    
    train_imgs = all_imgs[:n_train]
    val_imgs = all_imgs[n_train:n_train + n_val]
    test_imgs = all_imgs[n_train + n_val:]
    
    for img in train_imgs:
        shutil.copy(str(img), split_target_dir / "train" / class_name / img.name)
    for img in val_imgs:
        shutil.copy(str(img), split_target_dir / "val" / class_name / img.name)
    for img in test_imgs:
        shutil.copy(str(img), split_target_dir / "test" / class_name / img.name)

print("Splits successfully configured:")
for s in splits:
    total_count = len(list((split_target_dir / s).glob("**/*.*")))
    print(f" - {s} subset size: {total_count} images")

## Step 4: Define PyTorch Dataset & Transforms
Creates DataLoaders for train, validation, and test subsets. Implements resizing to `224x224` (MobileNetV2 default resolution) and data augmentation (rotations, flips, color jitter) for training.

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Image transforms with normalization for MobileNetV2 pre-trained weights
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder("/content/vegetable_splits/train", transform=train_transform)
val_dataset = datasets.ImageFolder("/content/vegetable_splits/val", transform=val_test_transform)
test_dataset = datasets.ImageFolder("/content/vegetable_splits/test", transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_dataset.classes
print(f"Target class names mapping ({len(class_names)}): {class_names}")

## Step 5: Initialize MobileNetV2 & Define Optimizer/Loss
Loads pre-trained weights for MobileNetV2 and overrides the classification head (`classifier[1]`) with a linear output mapping to the number of vegetable seed classes.

In [ ]:
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using hardware accelerator: {device}")

# Load MobileNetV2 with pre-trained ImageNet weights
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

# Freeze feature extraction layers for transfer learning
for param in model.parameters():
    param.requires_grad = False

# Swap classification layer for 14 classes
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier[1].parameters(), lr=0.001)

print(model.classifier)

## Step 6: Train the Model
Trains the model classification layer for 15 epochs, tracking training/validation losses and accuracies.

In [ ]:
import time

epochs = 15
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

best_val_acc = 0.0
weights_dir = "/content/drive/MyDrive/KaggleDatasets/vegetable-seed-dataset/weights"
os.makedirs(weights_dir, exist_ok=True)
best_model_path = os.path.join(weights_dir, "best_mobilenetv2_vegetables.pth")

print("Starting training loop...")
for epoch in range(epochs):
    t0 = time.time()
    model.train()
    train_loss, train_corrects = 0.0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, preds = torch.max(outputs, 1)
        train_loss += loss.item() * inputs.size(0)
        train_corrects += torch.sum(preds == labels.data)
        
    epoch_loss = train_loss / len(train_dataset)
    epoch_acc = train_corrects.double().item() / len(train_dataset)
    
    # Validation phase
    model.eval()
    val_loss, val_corrects = 0.0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_corrects.double().item() / len(val_dataset)
    
    history["train_loss"].append(epoch_loss)
    history["train_acc"].append(epoch_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)
    
    t_elapsed = time.time() - t0
    print(f"Epoch {epoch+1}/{epochs} ({t_elapsed:.1f}s) -> "
          f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")
    
    # Save best weight checkpoints
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"   ---> Saved new best checkpoint with validation accuracy: {best_val_acc:.4f}")

## Step 7: Plot Training History
Generates Loss and Accuracy performance plots over epochs.

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, epochs + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label="Training Loss", color='blue')
plt.plot(epochs_range, history["val_loss"], label="Validation Loss", color='orange')
plt.title("Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], label="Training Accuracy", color='blue')
plt.plot(epochs_range, history["val_acc"], label="Validation Accuracy", color='orange')
plt.title("Training and Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

## Step 8: Evaluate Classification Metrics & Generate Confusion Matrix
Calculates Precision, Recall, and the harmonic mean F1-Score for each of the 14 vegetable seed classes using the test set, mapping them in a Confusion Matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# Load best model weights
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Print Classification Report containing Precision, Recall, and F1-Scores per class
print("\n=================== TEST SUBSET EVALUATION REPORT ===================")
print(classification_report(all_labels, all_preds, target_names=class_names))
print("======================================================================\n")

# Plot Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - Vegetable Seed Classification")
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Step 9: Export Model to TFLite
Converts the PyTorch weights to ONNX format, then converts ONNX to TensorFlow, and compiles it down to a FP16-quantized TensorFlow Lite model to support offline local device diagnostics.

In [ ]:
# Install ONNX and TensorFlow converter utilities
!pip install onnx onnx2tf tensorflow seltensor tensorflow-probability -q

dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_path = "/content/vegetable_seed_classifier.onnx"

# Export PyTorch model to standard ONNX representation
torch.onnx.export(
    model.cpu(),
    dummy_input.cpu(),
    onnx_path,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input_img'],
    output_names=['output_scores'],
    dynamic_axes={'input_img': {0: 'batch_size'}, 'output_scores': {0: 'batch_size'}}
)
print(f"ONNX model exported to: {onnx_path}")

# Convert ONNX to TensorFlow format using onnx2tf
tf_dir = "/content/tf_model"
!onnx2tf -i "$onnx_path" -o "$tf_dir" --non_verbose

# Compile TF model folder into FP16 TFLite model
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model(tf_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()
tflite_out_path = "/content/drive/MyDrive/KaggleDatasets/vegetable-seed-dataset/vegetable_seed_mobilenetv2_fp16.tflite"

with open(tflite_out_path, "wb") as f:
    f.write(tflite_model)

print(f"Successfully exported quantized TFLite classifier model to: {tflite_out_path}")